In [1]:
# Imports.
import numpy as np;
import xarray as xr;
import matplotlib.pyplot as plt;
import h5_reader_xr as reader;
import gysela_utilities as gys_utils;
import phi2D_utilities as utils;
import phi2D_analytical as analytical;
from scipy.signal import find_peaks;

# Styling.
plt.style.use("ggplot");

In [ ]:
# Test inputs - delete when done!
directory_path = "/zhisongqu_data/seth/GYSELA/raw/batch_11.0/DN_REF_RH_AE_NR64NTHETA128NPHI8NVPAR512NMU64_Q1.5_NOPERT";
data_arrays = reader.fetch_phi2D_data(directory_path, parallelise = True);
dt_diag = reader.fetch_dt_diag(directory_path);
jacobian_dictionary = reader.fetch_jacobian(directory_path);

In [ ]:
def plot_gam_damping(phi2D_list, dt_diag, jacobian_dictionary, GAM_frequency, effective_radius = None):
	# We start with Phi_00. At this stage in the process flow, we operate with xarrays.
	fs_avg_time_series_naive = utils.generate_poloidally_averaged_time_series(phi2D_list, jacobian_dictionary);
	fs_avg_time_series_at_r = gys_utils.slice_at_effective_radius(fs_avg_time_series_naive, effective_radius);
	radially_averaged_time_series_naive = gys_utils.radial_average_2D(fs_avg_time_series_naive, jacobian_dictionary);

	# The values to be plotted.
	decay_signal = (fs_avg_time_series_at_r - radially_averaged_time_series_naive).values;
	time_range = np.arange(len(decay_signal)) * dt_diag;
	
	# Peak isolation.
	minimum_distance = 0.5 * (1 / GAM_frequency) * (1 / dt_diag);
	peak_indices, _ = find_peaks(decay_signal, distance = minimum_distance);
	peak_times = time_range[peak_indices];
	peak_values = decay_signal[peak_indices];

	# Fitting.
	log_peaks = np.log(peak_values);
	damping_rate, log_amplitude = np.polyfit(peak_times, log_peaks, 1);
	fitted_curve = np.exp(log_amplitude) * np.exp(damping_rate * time_range);

	# Figure plotting logic.
	_, ax = plt.subplots(figsize = (10, 6));
	ax.semilogy(time_range, np.abs(decay_signal), label = "Signal", color = "red", linestyle = "--");
	ax.plot(time_range, fitted_curve, label = "Fitted Exponential Decay", color = "blue", linestyle = "--");
	ax.scatter(peak_times, np.abs(peak_values), label = "Peaks", color = "black", marker = "o");
	ax.set_title(fr"$\gamma \Omega_i^{{-1}} = {damping_rate : .3e}$", fontsize = 16, pad = 15);
	ax.set_xlabel(r"time $\times \Omega_i$", fontsize = 16, labelpad = 10);
	ax.set_ylabel(r"$\phi_{00}(r_p, t) - \langle \phi_{00}(r) \rangle (t)$", fontsize = 16, labelpad = 10);
	ax.grid(True, which = "major", linestyle = ":", color = "gray", linewidth = 1);
	ax.tick_params(axis = "both", which = "major", labelsize = 14, direction = "in");
	plt.tight_layout();
	plt.show();
